In [ ]:

import numpy as np
import pandas as pd
import MDAnalysis as mda
import matplotlib.pyplot as plt
#import plotly.express as px
import scipy.stats as stats
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import pickle
import parmed
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


import matplotlib.ticker as ticker
from matplotlib.colors import TwoSlopeNorm
import io


In [ ]:
plt.style.use('./paper_style.mpl')

In [ ]:
## Size data

total_size_df = pd.DataFrame()

for i in ['3res','9res']:
    for j in ['equal','charged','aromatic','polar','nonpolar']:
        temp_df = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/size_data/{i}/{j}_size.csv', 
                              index_col=0)
        temp_df_2 = pd.DataFrame()
        temp_df_2[f'{i[0]}{j[0].upper()}'] = temp_df['prot-size'].values
        total_size_df = pd.concat([total_size_df, temp_df_2], axis=1)
        
for i in ['2_close','2_spread','3_close','3_spread']:
    if i == '2_close':
        name = 'GGGXGXGGG'
        num_stickers = 2
    elif i == '2_spread':
        name = 'GXGGGGGXG'
        num_stickers = 2
    elif i == '3_close':
        name = 'GGGXXXGGG'
        num_stickers = 3
    else:
        name = 'GXGGXGGXG'
        num_stickers = 3
        
        
    #num_stickers = 1
    temp_df = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/size_data/close_spread/{i}_size.csv',
                          index_col=0)
    temp_df_2 = pd.DataFrame()
    temp_df_2[f'{name}'] = temp_df['prot-size'].values / num_stickers
    total_size_df = pd.concat([total_size_df, temp_df_2], axis=1)


total_size_df

total_size_df = total_size_df.rename(columns={'3E':'3E', '3C':'3C', '3A':'3A', '3P':'3P',
       '3N':'3N', '9E':'9E', '9C':'9C', '9A':'9A',
       '9P':'9P', '9N':'9N', 'GGGXGXGGG':'X2','GXGGGGGXG':'X2S', 'GGGXXXGGG':'X3',
       'GXGGXGGXG':'X3S'})

## SASA data


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# DataFrame: total_size_df

systems = total_size_df.columns

rng = np.random.default_rng(42)

n_bootstrap = 4
sample_size = 300

# subplot layout
ncols = 5
nrows = int(np.ceil(len(systems) / ncols))

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(1.8*ncols, 1.6*nrows),
    constrained_layout=True,sharex=True,sharey=True
)

axes = axes.flatten()

for ax, system in zip(axes, systems):

    data = total_size_df[system].dropna().values

    for i in range(n_bootstrap):

        boot_sample = rng.choice(
            data,
            size=sample_size,
            replace=True
        )

        ax.hist(
            boot_sample,
            bins=10,
            alpha=0.5,
            #density=True,
            weights=np.ones(len(boot_sample)) / len(boot_sample),
            label=f'sample {i+1}'
        )

    ax.set_title(system)
    ax.set_xlabel('Cluster Size')
    ax.set_ylabel('Density')

axes[0].legend(fontsize=8)

# remove unused axes
for ax in axes[len(systems):]:
    fig.delaxes(ax)

plt.savefig("Figures/cluster_size_conver.png", dpi=400)


plt.show()

In [ ]:

total_ratio_df = pd.DataFrame()

# -------------------------
# 3res / 9res systems
# -------------------------
for i in ['3res']:
    for j in ['equal','charged','aromatic','polar','nonpolar']:
        
        df = pd.read_csv(f'data/{j}_cluster_water.csv')  # adjust if needed
        
        # compute ratio
        ratio = df['water_mass'] / df['protein_mass']
        
        temp_df = pd.DataFrame()
        temp_df[f'{i[0]}{j[0].upper()}'] = ratio.values
        
        total_ratio_df = pd.concat([total_ratio_df, temp_df], axis=1)
        
        
for i in ['9res']:
    for j in ['equal','charged','aromatic','polar','nonpolar']:
        
        df = pd.read_csv(f'data/{j}_cluster_water_9res.csv')  # adjust if needed
        
        # compute ratio
        ratio = df['water_mass'] / df['protein_mass']
        
        temp_df = pd.DataFrame()
        temp_df[f'{i[0]}{j[0].upper()}'] = ratio.values
        
        total_ratio_df = pd.concat([total_ratio_df, temp_df], axis=1)


# -------------------------
# close / spread systems
# -------------------------
for i in ['2_close','2_spread','3_close','3_spread']:
    
    if i == '2_close':
        name = 'GGGXGXGGG'
        num_stickers = 2
    elif i == '2_spread':
        name = 'GXGGGGGXG'
        num_stickers = 2
    elif i == '3_close':
        name = 'GGGXXXGGG'
        num_stickers = 3
    else:
        name = 'GXGGXGGXG'
        num_stickers = 3
        
        
    num_stickers = 1

    df = pd.read_csv(f'data/{i}_cluster_water_9res.csv')
    
    ratio = df['water_mass'] / df['protein_mass']
    
    temp_df = pd.DataFrame()
    temp_df[name] = ratio.values / num_stickers   # normalize like size
    
    total_ratio_df = pd.concat([total_ratio_df, temp_df], axis=1)


# -------------------------
# Rename columns (same as size)
# -------------------------
total_ratio_df = total_ratio_df.rename(columns={
    '3E':'3E', '3C':'3C', '3A':'3A', '3P':'3P', '3N':'3N',
    '9E':'9E', '9C':'9C', '9A':'9A', '9P':'9P', '9N':'9N',
    'GGGXGXGGG':'X2', 'GXGGGGGXG':'X2S',
    'GGGXXXGGG':'X3', 'GXGGXGGXG':'X3S'
})


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# DataFrame: total_size_df

systems = total_ratio_df.columns

rng = np.random.default_rng(42)

n_bootstrap = 4
sample_size = 200

# subplot layout
ncols = 5
nrows = int(np.ceil(len(systems) / ncols))

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(2.1*ncols, 1.7*nrows),
    constrained_layout=True,sharex=True,sharey=True
)

axes = axes.flatten()

for ax, system in zip(axes, systems):

    data = total_ratio_df[system].dropna().values

    for i in range(n_bootstrap):

        boot_sample = rng.choice(
            data,
            size=sample_size,
            replace=True
        )

        ax.hist(
            boot_sample,
            bins=10,
            alpha=0.5,
            #density=True,
            weights=np.ones(len(boot_sample)) / len(boot_sample),

            label=f'sample {i+1}'
        )

    ax.set_title(system)
    ax.set_xlabel('Water / Protein Mass Ratio')
    ax.set_ylabel('Density')

axes[0].legend(fontsize=8)

# remove unused axes
for ax in axes[len(systems):]:
    fig.delaxes(ax)

plt.savefig("Figures/water_protein_conver.png", dpi=400)


plt.show()

In [ ]:
## Water - Protein in a condenstae 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# =====================================================
# SYSTEMS
# =====================================================

systems = ['aromatic', 'equal', 'polar', 'nonpolar']

label_s = ['9A','9E','9P','9N']

# =====================================================
# FIGURE
# =====================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(9, 5),
    dpi=300,
    sharex=True
)

axes = axes.flatten()

protein_color = "#C44E52"
water_color = "#4C72B0"

for i,(ax1, system) in enumerate(zip(axes, systems)):

    # -------------------------------------------------
    # LOAD DATA
    # -------------------------------------------------

    df = pd.read_csv(
        f"condensate_center_rdf_protein_9_{system}.csv"
    )

    df_ow = pd.read_csv(
        f"condensate_center_rdf_water_9_{system}.csv"
    )

    # -------------------------------------------------
    # AVERAGE RDF
    # -------------------------------------------------

    avg_rdf = (
        df.groupby("r", as_index=False)["g_r"]
        .mean()
    )

    avg_rdf_ow = (
        df_ow.groupby("r", as_index=False)["g_r"]
        .mean()
    )

    # -------------------------------------------------
    # PROTEIN RDF (LEFT AXIS)
    # -------------------------------------------------

    line1 = ax1.plot(
        avg_rdf["r"],
        avg_rdf["g_r"],
        lw=1.0,
        color=protein_color,
        label="Protein"
    )

    ax1.fill_between(
        avg_rdf["r"],
        avg_rdf["g_r"],
        color=protein_color,
        alpha=0.15
    )

    ax1.set_ylabel("Protein $g(r)$",
                   color=protein_color,
                   fontsize=8)

    ax1.tick_params(axis='y',
                    colors=protein_color,
                    labelsize=7)

    # -------------------------------------------------
    # WATER RDF (RIGHT AXIS)
    # -------------------------------------------------

    ax2 = ax1.twinx()

    line2 = ax2.plot(
        avg_rdf_ow["r"],
        avg_rdf_ow["g_r"],
        lw=1.0,
        ls='--',
        color=water_color,
        label="Water"
    )

    ax2.fill_between(
        avg_rdf_ow["r"],
        avg_rdf_ow["g_r"],
        color=water_color,
        alpha=0.12
    )

    ax2.set_ylabel("Water $g(r)$",
                   color=water_color,
                   fontsize=8)

    ax2.tick_params(axis='y',
                    colors=water_color,
                    labelsize=7)

    # -------------------------------------------------
    # DECORATION
    # -------------------------------------------------

    ax1.axhline(
        y=1,
        color='gray',
        lw=1,
        ls=':',
        alpha=0.7
    )

    ax1.grid(
        alpha=0.25,
        linestyle='--',
        linewidth=0.5
    )

    ax1.set_title(label_s[i], fontsize=10)

# =====================================================
# X LABELS ONLY ON BOTTOM ROW
# =====================================================

for ax in axes[2:]:
    ax.set_xlabel(
        r"Distance from condensate center ($\AA$)",
        fontsize=8
    )

# =====================================================
# SINGLE LEGEND
# =====================================================

lines = line1 + line2
labels = [l.get_label() for l in lines]

fig.legend(
    lines,
    labels,
    loc="upper center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 1.02)
)

# =====================================================
# SAVE
# =====================================================

plt.tight_layout(rect=[0, 0, 1, 0.96])

plt.savefig(
    "Figures/rdf_condensate_center_all.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()